# 01. Vector DB 기초: PDF 로딩 → 청킹 (실습 1 · Step 0~2)

**프로젝트:** AI·반도체 정책 리포트 RAG (Dense → Hybrid → Rerank 검색 품질 비교)
**이 노트북의 목표:** PDF 16개를 읽어서 → 깨끗한 텍스트로 만들고 → 검색하기 좋은 크기(청크)로 잘라 → 파일로 저장한다.

```
[PDF 16개] → Step 1. 로딩·전처리 → [페이지 텍스트 + 꼬리표] → Step 2. 청킹 → [청크 약 2,200개] → outputs/chunks.jsonl
```

| No | 문서 | 발행처 | 성격 |
|---|---|---|---|
| 1 | SPRi AI Brief (2025.01 ~ 2026.06, 12개 호) | 소프트웨어정책연구소 | AI 산업 뉴스 |
| 2 | ETRI 인공지능(AI) 실행전략 (2020) | 한국전자통신연구원 | AI 국가 전략 |
| 3 | 글로벌 반도체 산업 전망 2026 | 삼일PwC | 반도체 시장 |
| 4 | AI 반도체 강국도약 가이드라인 (2025) | 반도체특별위원회 | 반도체 정책 |
| 5 | AI 반도체 기술 및 산업 동향 (2024.07) | KDB미래전략연구소 | 반도체 기술 |

## Step 0. 환경 확인

이 셀은 **"실습에 필요한 도구가 다 깔려 있나"**를 점검하는 셀이다. 하나라도 ❌가 뜨면 터미널에서 `uv sync`를 다시 해보고, 커널이 `.venv`로 잡혀 있는지 확인한다.

Qdrant는 Step 5부터 쓰니까, 지금 연결이 안 돼도 Step 1~2 진행에는 문제없다.

In [1]:
# ─────────────────────────────────────────────────────────────
# Step 0. 실습에 필요한 라이브러리가 모두 설치됐는지 하나씩 확인
# ─────────────────────────────────────────────────────────────
import importlib
import logging
import sys

# FlagEmbedding을 불러올 때 torch가 "Redirects are currently not supported in MacOs" 경고를 빨간 글씨로 띄운다.
# macOS에서 분산학습 기능이 없다는 안내일 뿐 실습과 무관해서 숨긴다.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

print(f"Python 버전: {sys.version.split()[0]}  (3.11 권장)")
print(f"실행 위치  : {sys.executable}")   # 경로에 .venv 가 들어 있으면 가상환경 커널이 맞게 잡힌 것
print()

# (import 할 때 쓰는 이름, 화면에 보여줄 이름, 어디서 쓰는지)
REQUIRED = [
    ("pymupdf",                  "PyMuPDF",                  "Step 1 · PDF에서 글자 꺼내기"),
    ("langchain_text_splitters", "LangChain Text Splitters", "Step 2 · 텍스트 자르기"),
    ("numpy",                    "NumPy",                    "Step 3 · 벡터(숫자 배열) 다루기"),
    ("FlagEmbedding",            "FlagEmbedding",            "Step 3 · BGE-M3 임베딩 / 실습2 Reranker"),
    ("faiss",                    "FAISS",                    "Step 4 · 임시 벡터 검색"),
    ("qdrant_client",            "Qdrant Client",            "Step 5 · 벡터 DB 서버 연결"),
    ("bm25s",                    "bm25s",                    "실습 2 · 키워드 검색"),
    ("ragas",                    "RAGAS",                    "실습 2 · 품질 평가"),
]

for module_name, label, usage in REQUIRED:
    try:
        importlib.import_module(module_name)
        print(f"✅ {label:<26} {usage}")
    except ImportError:
        print(f"❌ {label:<26} {usage}  ← 설치 안 됨 (터미널에서 `uv sync`)")

Python 버전: 3.11.15  (3.11 권장)
실행 위치  : /Users/simhyeok/Desktop/vectordb-rag/.venv/bin/python



✅ PyMuPDF                    Step 1 · PDF에서 글자 꺼내기
✅ LangChain Text Splitters   Step 2 · 텍스트 자르기
✅ NumPy                      Step 3 · 벡터(숫자 배열) 다루기


✅ FlagEmbedding              Step 3 · BGE-M3 임베딩 / 실습2 Reranker
✅ FAISS                      Step 4 · 임시 벡터 검색


✅ Qdrant Client              Step 5 · 벡터 DB 서버 연결
✅ bm25s                      실습 2 · 키워드 검색


✅ RAGAS                      실습 2 · 품질 평가


In [2]:
# ─────────────────────────────────────────────────────────────
# Step 0-2. Qdrant(Docker) 서버가 켜져 있는지 확인
#   - Qdrant = 벡터를 보관하는 "창고 서버". Docker로 띄워 둔다.
#   - 안 켜져 있으면 터미널에서:  docker start qdrant
#     (처음이라면 README의 docker run 명령어 실행)
# ─────────────────────────────────────────────────────────────
try:
    from qdrant_client import QdrantClient

    client = QdrantClient(host="localhost", port=6333, timeout=3, check_compatibility=False)
    print("✅ Qdrant 연결 성공:", client.get_collections())
    # CollectionsResponse(collections=[]) → 아직 창고가 비어 있다는 뜻 (정상)
except Exception as e:
    print("⚠️ Qdrant에 연결할 수 없음 (Step 5 전까지만 켜두면 됨)")
    print("   원인:", type(e).__name__)

✅ Qdrant 연결 성공: collections=[]


## Step 1. PDF 로딩 + 전처리

PDF는 사람이 보기 좋은 "종이"라서 글자 말고도 디자인 요소가 잔뜩 섞여 있다. 컴퓨터가 검색하려면 **순수 텍스트**만 필요하다.

교재 코드는 PDF 1개만 읽지만, 여기서는 16개를 모두 읽고 아래 4가지를 추가로 처리한다.

| 처리 | 왜 필요한가 |
|---|---|
| ① 16개 전부 로딩 | 통합 질문은 여러 문서를 봐야 답이 나온다 |
| ② 꼬리표(메타데이터) 붙이기 | "반도체 문서만", "2026년 것만" 같은 필터 검색을 하려면 미리 달아둬야 한다 |
| ③ 머리글·꼬리글 제거 | `SPRi AI Brief / 2025년 10월호` 같은 줄이 모든 페이지에 반복돼서 검색 노이즈가 된다 |
| ④ 한글 정규화(NFC) | Mac은 한글 파일명을 자모 단위로 쪼개 저장해서, 나중에 이름으로 필터하면 0건이 나올 수 있다 |

In [3]:
# ─────────────────────────────────────────────────────────────
# Step 1-1. 라이브러리 불러오기 + 경로 설정
# ─────────────────────────────────────────────────────────────
import re             # 정규표현식: "숫자만 있는 줄" 같은 패턴 찾기
import json           # 결과를 파일로 저장할 때 사용
import unicodedata    # 한글 정규화(NFC)
from pathlib import Path
from collections import Counter

import pymupdf        # PDF에서 글자를 꺼내는 도구 (예전 이름: fitz)
import pandas as pd   # 결과를 표로 예쁘게 보기 위해 사용

# 이 노트북은 notebooks/ 폴더 안에 있으므로, 한 칸 위(..)가 프로젝트 루트
PROJECT_ROOT = Path("..").resolve()
PDF_DIR = PROJECT_ROOT / "data" / "pdfs"     # PDF 16개가 들어 있는 곳
OUT_DIR = PROJECT_ROOT / "outputs"           # 청크·임베딩 결과를 저장할 곳
OUT_DIR.mkdir(exist_ok=True)                 # 폴더가 없으면 새로 만든다

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"PDF 폴더: {PDF_DIR}")
print(f"찾은 PDF: {len(pdf_files)}개 (16개가 정상)")

PDF 폴더: /Users/simhyeok/Desktop/vectordb-rag/data/pdfs
찾은 PDF: 16개 (16개가 정상)


In [4]:
# ─────────────────────────────────────────────────────────────
# Step 1-2. 문서별 꼬리표(메타데이터) 규칙 정하기
#
# 파일명이 "1. SPRi ...", "2. ETRI ..." 처럼 번호로 시작하니까
# 맨 앞 번호만 보고 어떤 문서인지 판단한다. (한글 비교를 피하는 안전한 방법)
#
# 나중에 Qdrant에서 이렇게 쓸 예정:
#   category = "반도체"   → 반도체 문서에서만 검색
#   year >= 2026          → 최신 자료에서만 검색
# ─────────────────────────────────────────────────────────────
DOC_INFO = {
    "1": {"publisher": "SPRi",            "category": "AI동향"},
    "2": {"publisher": "ETRI",            "category": "AI전략"},
    "3": {"publisher": "삼일PwC",          "category": "반도체"},
    "4": {"publisher": "반도체특별위원회",   "category": "반도체"},
    "5": {"publisher": "KDB미래전략연구소", "category": "반도체"},
}

# SPRi는 파일명 끝에 "_202510" 처럼 연월이 있어서 자동으로 뽑는다.
# 나머지는 파일명에 날짜가 없어서 발행 연도를 직접 적어 둔다.
FIXED_YEAR = {"2": 2020, "3": 2025, "4": 2025, "5": 2024}


def get_meta(pdf_path: Path) -> dict:
    """PDF 파일 경로를 받아서, 그 문서의 꼬리표(dict)를 돌려준다."""
    # Mac은 한글을 "ㅅ+ㅏ+ㅁ" 처럼 쪼개서(NFD) 저장한다.
    # NFC로 합쳐두지 않으면 나중에 "삼일PwC"로 필터할 때 0건이 나올 수 있다.
    name = unicodedata.normalize("NFC", pdf_path.stem)   # stem = 확장자(.pdf) 뺀 파일명

    doc_no = name.split(".")[0]                           # "1. SPRi..." → "1"

    # 파일명 끝이 "_YYYYMM" 형태면 앞 4자리를 연도로 사용
    match = re.search(r"_(\d{4})\d{2}$", name)
    year = int(match.group(1)) if match else FIXED_YEAR[doc_no]

    return {"source": name, "year": year, **DOC_INFO[doc_no]}


# 잘 되는지 첫 번째, 마지막 파일로 테스트
print(get_meta(pdf_files[0]))
print(get_meta(pdf_files[-1]))

{'source': '1. SPRi AI Brief_10월호_산업동향_202510', 'year': 2025, 'publisher': 'SPRi', 'category': 'AI동향'}
{'source': '5. AI 반도체 기술 및 산업 동향', 'year': 2024, 'publisher': 'KDB미래전략연구소', 'category': '반도체'}


In [5]:
# ─────────────────────────────────────────────────────────────
# Step 1-3. 전처리 함수: 반복되는 머리글/꼬리글 지우기
#
# 아이디어: "한 문서 안에서 30% 이상의 페이지에 똑같이 나오는 짧은 줄"은
#          본문이 아니라 머리글·꼬리글일 확률이 높다.
#   예) "SPRi AI Brief", "AI 반도체 강국도약 가이드라인", "Samil PwC"
#
# 문서마다 머리글을 하나하나 적어주지 않아도 자동으로 찾아내는 방식이라
# 새 PDF가 추가돼도 그대로 쓸 수 있다.
# ─────────────────────────────────────────────────────────────
def find_repeated_lines(raw_pages: list[str], ratio: float = 0.3) -> set[str]:
    """여러 페이지에 반복해서 등장하는 짧은 줄(40자 이하)을 찾는다."""
    counter = Counter()
    for page_text in raw_pages:
        # set으로 묶는 이유: 한 페이지 안에서 같은 줄이 2번 나와도 1번으로 센다
        lines = {line.strip() for line in page_text.splitlines() if 0 < len(line.strip()) <= 40}
        counter.update(lines)

    threshold = len(raw_pages) * ratio      # 예: 100페이지 문서면 30페이지 이상 반복된 줄
    return {line for line, count in counter.items() if count >= threshold}


def clean_text(text: str, repeated: set[str]) -> str:
    """한 페이지 텍스트에서 노이즈 줄을 지우고 깨끗한 본문만 남긴다."""
    kept = []
    for line in text.splitlines():
        s = line.strip()
        if not s:                       # 빈 줄
            continue
        if s in repeated:               # 반복되는 머리글/꼬리글
            continue
        if "●●●" in s:                  # ETRI 페이지 머리글 ("www.etri.re.kr ●●● 40")
            continue
        if re.fullmatch(r"[\d\s\-–|]+", s):   # 숫자·기호만 있는 줄 = 페이지 번호
            continue
        kept.append(s)
    return "\n".join(kept)


print("✅ 전처리 함수 준비 완료: find_repeated_lines(), clean_text()")

✅ 전처리 함수 준비 완료: find_repeated_lines(), clean_text()


In [6]:
# ─────────────────────────────────────────────────────────────
# Step 1-4. (눈으로 확인) 전처리 전/후 비교
#   SPRi 10월호 6페이지를 예시로, 어떤 줄이 지워지는지 직접 본다.
# ─────────────────────────────────────────────────────────────
sample_pdf = next(p for p in pdf_files if "202510" in p.name)
raw_pages = [page.get_text() for page in pymupdf.open(sample_pdf)]
repeated = find_repeated_lines(raw_pages)

print("🔁 이 문서에서 '반복 줄'로 판단된 것들:")
for line in sorted(repeated):
    print("   -", line)

print("\n" + "=" * 60)
print("[전처리 전] 앞부분 300자")
print(raw_pages[5][:300])
print("=" * 60)
print("[전처리 후] 앞부분 300자")
print(clean_text(raw_pages[5], repeated)[:300])

🔁 이 문서에서 '반복 줄'로 판단된 것들:
   - 2025년 10월호
   - KEY Contents
   - SPRi AI Brief
   - 기술･연구
   - 기업･산업
   - 인력･교육
   - 정책･법제

[전처리 전] 앞부분 300자
SPRi AI Brief
2025년 10월호
4
영국 과학혁신기술부, 2024년 영국 AI 산업 현황 분석 보고서 발간
n 영국 정부가 발표한 2024년 영국 AI 산업 분석 보고서에 따르면 2024년 기준 영국 AI 기업 수는 
전년 대비 58% 증가했으며, 총매출은 68% 증가해 영국 AI 산업의 빠른 성장세를 입증
n 2024년 영국의 AI 전문 기업은 한화 약 5조 원이 넘는 투자를 유치했으며, 2019년 이후 영국 내 
AI 투자 프로젝트의 5분의 1이 2024년에 시작되는 등, 영국은 AI 투자 선호지로 부상하는 추세
K
[전처리 후] 앞부분 300자
영국 과학혁신기술부, 2024년 영국 AI 산업 현황 분석 보고서 발간
n 영국 정부가 발표한 2024년 영국 AI 산업 분석 보고서에 따르면 2024년 기준 영국 AI 기업 수는
전년 대비 58% 증가했으며, 총매출은 68% 증가해 영국 AI 산업의 빠른 성장세를 입증
n 2024년 영국의 AI 전문 기업은 한화 약 5조 원이 넘는 투자를 유치했으며, 2019년 이후 영국 내
AI 투자 프로젝트의 5분의 1이 2024년에 시작되는 등, 영국은 AI 투자 선호지로 부상하는 추세
£ 2024년 영국 AI 산업, 기업 수와 총매출, 일


In [7]:
# ─────────────────────────────────────────────────────────────
# Step 1-5. 16개 PDF 전체 로딩
#
# 결과물 pages = [
#   {"text": "본문...", "page": 3, "source": "1. SPRi ...", "year": 2025,
#    "publisher": "SPRi", "category": "AI동향"},
#   ...
# ]
# ─────────────────────────────────────────────────────────────
def load_pdfs(pdf_dir: Path, min_chars: int = 100) -> list[dict]:
    pages = []
    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        meta = get_meta(pdf_path)                         # 이 문서의 꼬리표
        doc = pymupdf.open(pdf_path)
        raw_pages = [page.get_text() for page in doc]     # 페이지별 원본 텍스트
        repeated = find_repeated_lines(raw_pages)         # 이 문서만의 머리글 목록

        for i, raw in enumerate(raw_pages):
            text = unicodedata.normalize("NFC", clean_text(raw, repeated))

            # 100자 미만 = 표지, 목차 구분 페이지, 그림만 있는 페이지 → 검색에 쓸모없어서 제외
            if len(text) < min_chars:
                continue

            pages.append({
                "text": text,
                "page": i + 1,        # enumerate는 0부터 세니까 +1 해서 사람 기준 페이지 번호로
                **meta,               # source, year, publisher, category 펼쳐 넣기
            })
    return pages


pages = load_pdfs(PDF_DIR)
print(f"✅ 로딩 완료: {len(pages)}페이지 (참고값: 762)")

✅ 로딩 완료: 762페이지 (참고값: 762)


In [8]:
# ─────────────────────────────────────────────────────────────
# Step 1-6. 문서별로 몇 페이지씩 들어왔는지 표로 확인
#   → 어떤 문서가 데이터의 대부분을 차지하는지 감을 잡는다
# ─────────────────────────────────────────────────────────────
df_pages = pd.DataFrame(pages)
summary = (
    df_pages.groupby(["publisher", "category"])
    .agg(파일수=("source", "nunique"), 페이지수=("page", "count"), 총글자수=("text", lambda s: s.str.len().sum()))
    .sort_values("페이지수", ascending=False)
)
summary["비중(%)"] = (summary["총글자수"] / summary["총글자수"].sum() * 100).round(1)
summary

,,파일수,페이지수,총글자수,비중(%)
publisher,category,,,,
SPRi,AI동향,12,280,420808,48.7
ETRI,AI전략,1,248,221931,25.7
반도체특별위원회,반도체,1,109,123961,14.4
삼일PwC,반도체,1,100,75965,8.8
KDB미래전략연구소,반도체,1,25,20841,2.4


### 📝 Step 1 관찰 메모

- 전체 866페이지 중 100자 미만 페이지(표지, 목차 구분, 그림 위주)가 걸러져서 약 760페이지가 남는다.
- **SPRi + ETRI가 전체의 약 3/4**이다. 반도체 문서는 1/4 정도라서, 반도체 질문을 하면 AI 뉴스가 섞여 나올 가능성이 높다. 실습 2에서 Dense 실패 케이스를 볼 때 이 점을 떠올리자.
- 삼일PwC 문서는 PDF 구조 때문에 **띄어쓰기가 사라진 채로** 추출된다. 아래 셀에서 직접 확인해본다.

In [9]:
# ─────────────────────────────────────────────────────────────
# Step 1-7. (눈으로 확인) 삼일PwC 문서의 띄어쓰기 문제
#   띄어쓰기 없이 12글자 이상 이어진 한글 덩어리가 몇 개인지 문서별로 센다.
#   → 실습 2의 BM25(띄어쓰기 기준으로 단어를 자르는 검색)에 영향을 준다.
# ─────────────────────────────────────────────────────────────
glued = Counter()
for p in pages:
    glued[p["publisher"]] += len(re.findall(r"[가-힣]{12,}", p["text"]))
print("띄어쓰기 없이 12자 이상 붙은 한글 덩어리 수:", dict(glued))

example = next(p for p in pages if p["publisher"] == "삼일PwC" and re.search(r"[가-힣]{20,}", p["text"]))
print(f"\n예시 ({example['source']} p.{example['page']}):")
print(re.search(r"[가-힣]{20,}", example["text"]).group())

띄어쓰기 없이 12자 이상 붙은 한글 덩어리 수: {'SPRi': 17, 'ETRI': 5, '삼일PwC': 1075, '반도체특별위원회': 0, 'KDB미래전략연구소': 0}

예시 (3. 글로벌 반도체 산업 전망 2026_삼일회계법인 p.3):
각국의내수생산투자확대에힘입어그어느때보다빠르게재편되고


## Step 2. 청킹 (Chunking)

페이지 하나도 LLM에 통째로 주기엔 크고, 검색 단위로도 너무 뭉툭하다. 그래서 **약 500자 조각(청크)**으로 자른다.

- `chunk_size=500` → 한 조각 최대 500**글자** (교재의 "300~500 토큰"과 단위가 다르다는 점 주의)
- `chunk_overlap=50` → 조각끼리 50자씩 겹쳐서 경계에서 문장이 끊기지 않게
- `separators` → 문단 → 줄바꿈 → 마침표 → 띄어쓰기 순으로 "자연스러운 곳"을 먼저 찾아 자른다 (Recursive 방식)

In [10]:
# ─────────────────────────────────────────────────────────────
# Step 2-1. 자르는 도구(splitter) 만들기
# ─────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    # 앞에서부터 우선순위: 이 기호로 먼저 잘라보고, 그래도 500자 넘으면 다음 기호로
    separators=["\n\n",   # ① 문단 경계
                "\n",     # ② 줄바꿈
                ".",      # ③ 문장 끝
                " "],     # ④ 단어 사이 (최후의 수단)
)

print(f"✅ splitter 준비 완료: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")

✅ splitter 준비 완료: chunk_size=500, chunk_overlap=50


In [11]:
# ─────────────────────────────────────────────────────────────
# Step 2-2. 모든 페이지를 청크로 자르기
#
# 핵심: 페이지 단위로 자르기 때문에 한 청크가 두 페이지에 걸치지 않는다.
#       → 청크마다 "몇 페이지에서 왔는지" 정확히 남길 수 있다.
#
# chunk_id = 0, 1, 2, ... 순서 번호
#   ★ 이 번호가 앞으로 모든 걸 잇는 공통 열쇠다.
#     chunks[i] ↔ embeddings[i] ↔ FAISS 결과 i ↔ Qdrant id i ↔ BM25 결과 i
#     그래서 청크 설정을 바꾸면 임베딩부터 전부 다시 만들어야 한다.
# ─────────────────────────────────────────────────────────────
chunks = []
for p in pages:
    meta = {k: v for k, v in p.items() if k != "text"}   # 페이지의 꼬리표만 복사
    for piece in splitter.split_text(p["text"]):
        chunks.append({**meta, "text": piece})          # 청크마다 같은 꼬리표를 붙인다

for i, c in enumerate(chunks):
    c["chunk_id"] = i

print(f"✅ 청크 생성 완료: {len(chunks)}개 (참고값: 약 2,200개)")

✅ 청크 생성 완료: 2197개 (참고값: 약 2,200개)


In [12]:
# ─────────────────────────────────────────────────────────────
# Step 2-3. 청크 통계 확인
# ─────────────────────────────────────────────────────────────
df_chunks = pd.DataFrame(chunks)
df_chunks["length"] = df_chunks["text"].str.len()

print("📏 청크 길이(글자 수) 분포")
print(df_chunks["length"].describe().round(0).to_string())
print()

print("📚 문서별 청크 수")
print(df_chunks["publisher"].value_counts().to_string())
print()

# HBM이 들어간 청크가 얼마나 되는지 → 실습 질문 "HBM이란?"의 정답 후보가 몇 개인지
hbm = df_chunks[df_chunks["text"].str.contains("HBM")]
print(f"🔎 'HBM'이 들어간 청크: {len(hbm)}개 / 전체 {len(df_chunks)}개 ({len(hbm) / len(df_chunks) * 100:.1f}%)")
print(hbm["publisher"].value_counts().to_string())

📏 청크 길이(글자 수) 분포
count    2197.0
mean      407.0
std       122.0
min        18.0
25%       370.0
50%       465.0
75%       487.0
max       500.0

📚 문서별 청크 수
publisher
SPRi          1040
ETRI           588
반도체특별위원회       310
삼일PwC          205
KDB미래전략연구소      54

🔎 'HBM'이 들어간 청크: 69개 / 전체 2197개 (3.1%)
publisher
반도체특별위원회      44
KDB미래전략연구소    11
삼일PwC          9
ETRI           4
SPRi           1


In [13]:
# ─────────────────────────────────────────────────────────────
# Step 2-4. (눈으로 확인) 오버랩이 실제로 어떻게 생겼는지
#
# 같은 페이지에서 나온 연속된 두 청크(a → b)를 보고,
# b의 앞부분이 a의 끝부분에 그대로 들어 있으면 "겹친(오버랩된)" 것이다.
#
# ※ 재밌는 점: chunk_overlap=50은 "최대 50자까지 겹칠 수 있다"는 뜻이지
#    항상 겹친다는 뜻이 아니다. Recursive 방식은 문단·줄바꿈 같은 깔끔한 경계에서
#    잘리면 굳이 겹치지 않는다. 얼마나 겹치는지 직접 세어 본다.
# ─────────────────────────────────────────────────────────────
def overlap_text(a: str, b: str, min_len: int = 10) -> str:
    """a의 '끝부분'과 b의 '앞부분'이 똑같이 겹치는 가장 긴 문자열을 찾는다. 없으면 빈 문자열."""
    # 긴 길이부터 하나씩 줄여가며 "a가 b의 앞 k글자로 끝나는지" 확인
    for k in range(min(len(a), len(b), CHUNK_OVERLAP), min_len - 1, -1):
        if a.endswith(b[:k]):
            return b[:k]
    return ""


pairs, overlapped, example = 0, 0, None
for a, b in zip(chunks, chunks[1:]):
    if (a["source"], a["page"]) != (b["source"], b["page"]):   # 다른 페이지면 애초에 이어진 조각이 아님
        continue
    pairs += 1
    ov = overlap_text(a["text"], b["text"])
    if ov:
        overlapped += 1
        if example is None and "HBM" in a["text"] + b["text"]:
            example = (a, b, ov)

print(f"같은 페이지의 연속 청크 쌍: {pairs}개 중 실제로 겹친 쌍: {overlapped}개 ({overlapped / pairs * 100:.0f}%)")

if example:
    a, b, ov = example
    print(f"\n[청크 {a['chunk_id']}] {a['source']} p.{a['page']}")
    print("  ...끝부분 →", a["text"][-100:].replace("\n", " "))
    print(f"[청크 {b['chunk_id']}] 다음 조각")
    print("  시작부분 →", b["text"][:100].replace("\n", " "), "...")
    print(f"\n👉 겹친 부분({len(ov)}자): 「{ov.strip()}」")

같은 페이지의 연속 청크 쌍: 1435개 중 실제로 겹친 쌍: 811개 (57%)

[청크 1243] 2. ETRI 인공지능(AI) 실행전략 p.108
  ...끝부분 → 적층기술 연구가 활발히 전개되고 있음 - 대표적 기술로는 마이크론에서 선도하는 HMC(Hybrid Memory Cube)와 삼성전자와 SK하이닉스에서 개발 중인 HBM이 있음38)
[청크 1244] 다음 조각
  시작부분 → SK하이닉스에서 개발 중인 HBM이 있음38) 35) 연세대 ICSL, https://web.yonsei.ac.kr/icsl/research.html 36) AI타임스,　“AI 가 ...

👉 겹친 부분(25자): 「SK하이닉스에서 개발 중인 HBM이 있음38)」


In [14]:
# ─────────────────────────────────────────────────────────────
# Step 2-5. 청크 하나를 통째로 열어보기 (Qdrant에 저장될 모양 그대로)
# ─────────────────────────────────────────────────────────────
print(json.dumps(chunks[hbm.index[0]], ensure_ascii=False, indent=2))

{
  "page": 13,
  "source": "1. SPRi AI Brief_7월호_산업동향_202507",
  "year": 2025,
  "publisher": "SPRi",
  "category": "AI동향",
  "text": "n 이번 행사에는 오픈AI(OpenAI), 메타(Meta), 마이크로소프트(Microsoft) 등 주요 AI 기업들도 참여해\nAI 모델 개발을 위한 AMD와의 협력 현황을 논의\n∙특히 AMD는 오픈AI를 차세대 칩 MI450의 핵심 고객이자 MI450 개발 시 차세대 학습 및 추론\n요구사항에 대한 중요 피드백을 제공한 초기 설계 파트너로 소개\n∙MI450을 포함한 MI400 시리즈 GPU는 최대 432GB의 고대역폭 메모리(HBM4) 용량을 제공하고 4비트\n부동소수점 연산(FP4) 기준 40페타플롭스(초당 40,000조)의 연산을 수행\n출처 | AMD, AMD Unveils Vision for an Open AI Ecosystem, Detailing New Silicon, Software and Systems at Advancing AI 2025, 2025.06.12.",
  "chunk_id": 898
}


In [15]:
# ─────────────────────────────────────────────────────────────
# Step 2-6. 결과 저장 → outputs/chunks.jsonl
#
# jsonl = 한 줄에 청크 하나씩 JSON으로 저장하는 형식
# 왜 저장하나?
#   - 다음 노트북(Step 3 임베딩)에서 PDF를 다시 읽을 필요 없이 바로 불러오기 위해
#   - 임베딩 순서와 청크 순서가 어긋나지 않도록 "기준 파일"을 하나 고정해 두기 위해
# ─────────────────────────────────────────────────────────────
CHUNKS_PATH = OUT_DIR / "chunks.jsonl"

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")   # ensure_ascii=False → 한글이 그대로 저장됨

# 저장이 잘 됐는지 다시 읽어서 개수 확인
with open(CHUNKS_PATH, encoding="utf-8") as f:
    reloaded = [json.loads(line) for line in f]

print(f"💾 저장 완료: {CHUNKS_PATH}")
print(f"   다시 읽은 개수: {len(reloaded)}개  (위 청크 수와 같으면 OK)")

💾 저장 완료: /Users/simhyeok/Desktop/vectordb-rag/outputs/chunks.jsonl
   다시 읽은 개수: 2197개  (위 청크 수와 같으면 OK)


## ✅ Step 0~2 체크포인트

- [ ] Step 0: 라이브러리 모두 ✅, Qdrant 연결 확인 (Step 5 전까지)
- [ ] Step 1: PDF 16개 로딩, 약 760페이지
- [ ] Step 2: 청크 약 2,200개, `outputs/chunks.jsonl` 저장

### 다음: `02_embedding_faiss_qdrant.ipynb`
`chunks.jsonl`을 불러와서 → BGE-M3로 임베딩(결과는 `embeddings.npy`로 캐시) → FAISS 검색 → Qdrant 저장 → 메타데이터 필터 검색